# Sales Data Analysis & ETL Pipeline

## Introducción
Este proyecto analiza datos de ventas provenientes de múltiples archivos Excel con el objetivo de identificar patrones, tendencias y oportunidades de mejora.

**Cargar datos**

In [ ]:
#Cargar librerías
import pandas as pd
import glob

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
#Función de carga de archivos
def cargar_datos(ruta):
    archivos = glob.glob(ruta + "Sales*.csv")
    df_lista = []

    for archivo in archivos:
        df = pd.read_csv(archivo, low_memory=False)  
        df_lista.append(df)

    df_final = pd.concat(df_lista, ignore_index=True)
    return df_final

In [ ]:
df = cargar_datos("data/sample_data.csv")
df.head()

In [ ]:
#Validación
def cargar_datos(ruta):
    archivos = glob.glob(ruta + "Sales*.csv")

    if not archivos:
        print("No se encontraron archivos")
        return None

    df_lista = []

    for archivo in archivos:
        df = pd.read_csv(archivo)
        df_lista.append(df)

    df_final = pd.concat(df_lista, ignore_index=True)
    return df_final

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
#Eliminar valores nulos
df = df.drop(columns=["invoice_type", "sales_amount", "sales_amount.1"])

In [ ]:
#Analizar columnas vacías
df["quantity"].describe()
df["Sector"].value_counts(dropna=False)

In [ ]:
#Reemplazar NaN por "No es pecificado"
df["Sector"] = df["Sector"].fillna("No especificado")

La variable “Sector” presenta un alto porcentaje de valores faltantes (~95%). Sin embargo, se decidió conservarla, imputando los valores nulos como “No especificado”, ya que aporta información relevante sobre segmentos específicos de clientes.

In [ ]:
df["columna 4"] = df["columna 4"].fillna(0)

In [ ]:
df["columna 1"] = df["columna 2"].fillna("No especificado")

In [ ]:
df = df.dropna(subset=["columna 3"])

In [ ]:
df = df.dropna(subset=["columna 5"])

In [ ]:
df = df.dropna(subset=["columna 6"])

Se realizó un tratamiento diferenciado de valores faltantes según la naturaleza de cada variable. Las columnas completamente nulas fueron eliminadas, las variables categóricas se imputaron como “No especificado”, las variables numéricas con ausencia lógica se rellenaron con cero, y las columnas con pocos valores faltantes se limpiaron eliminando registros incompletos.

**Análisis**

In [ ]:
#Ventas totales
df["columna 7"].sum()

In [ ]:
#Ventas por mes 
df.groupby("Mes")["columna 7"].sum()

In [ ]:
#Ventas por producto
df.groupby("Articulo")["columna 7"].sum().sort_values(ascending=False).head(10)

In [ ]:
#Ventas por categoría
df.groupby("category name")["columna 7"].sum().sort_values(ascending=False)

In [ ]:
#Ventas por sector
df.groupby("Sector")["columna 7"].sum()

Una vez realizada la limpieza de datos, se procedió a analizar el comportamiento de las ventas, identificando tendencias por mes, productos y segmentos de clientes.

In [ ]:
#Ventas por mes por año
df["fecha"] = pd.to_datetime(df["Año"].astype(str) + "-" + df["Mes"].astype(str))

df.groupby("fecha")["columna 7"].sum().sort_index()

In [ ]:
#Visualización
df.groupby("fecha")["columna 7"].sum().plot()

Se analizó la evolución de las ventas a lo largo del tiempo considerando año y mes, lo que permitió identificar tendencias, estacionalidad y cambios en el comportamiento de ventas.

Se observa que las ventas presentan un comportamiento volátil a lo largo del tiempo, sin una tendencia de crecimiento sostenido. Se identifican picos significativos en ciertos periodos, lo que podría estar asociado a eventos comerciales o clientes específicos. Asimismo, se detectan caídas abruptas que sugieren dependencia de factores puntuales. Finalmente, se identificó un valor atípico en el último periodo, el cual corresponde a datos incompletos y fue excluido del análisis.

In [ ]:
df.groupby("category name")["columna 7"].sum().sort_values(ascending=False)

Se observa que la mayor parte de los ingresos proviene de una categoría principal de productos, lo que indica que el negocio depende significativamente de este segmento. Esto representa tanto una fortaleza —al ser un producto clave— como un riesgo potencial por la concentración de ingresos.

In [ ]:
df.groupby("category name")["columna 7"].sum() / df["columna 7"].sum()

In [ ]:
df_cat = df[df["category name"] == "name 1"]

In [ ]:
#Crear serie de tiempo
df_ts = df_cat.groupby("fecha")["quantity shipped"].sum().sort_index()

In [ ]:
df_ts.plot()

In [ ]:
#Crear variables
df_ts = df_cat.groupby("fecha")["quantity shipped"].sum().to_frame()

df_ts["lag1"] = df_ts["quantity shipped"].shift(1)
df_ts["lag2"] = df_ts["quantity shipped"].shift(2)
df_ts["lag3"] = df_ts["quantity shipped"].shift(3)

df_ts = df_ts.dropna()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = df_ts[["lag1", "lag2", "lag3"]]
y = df_ts["quantity shipped"]

model = RandomForestRegressor()
model.fit(X, y)

In [ ]:
y_pred = model.predict(X)

In [ ]:
#Separar datos
train = df_ts.iloc[:-6]   # todo menos últimos 6 meses
test = df_ts.iloc[-6:]    # últimos 6 meses

In [ ]:
#Definir variables
X_train = train[["lag1", "lag2", "lag3"]]
y_train = train["quantity shipped"]

X_test = test[["lag1", "lag2", "lag3"]]
y_test = test["quantity shipped"]

In [ ]:
#Entrenar modelo
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

In [ ]:
#Predecir
y_pred = model.predict(X_test)

In [ ]:
#Evaluar
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
mae

In [ ]:
#Gráfica
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(test.index, y_test, label="Real")
plt.plot(test.index, y_pred, label="Predicción")

plt.legend()
plt.title("Forecast vs Real")
plt.show()

El modelo presenta dificultades para capturar picos de demanda asociados a eventos atípicos, como compras extraordinarias de clientes específicos. Esto sugiere que el comportamiento de la serie no es completamente estacionario y depende de factores externos no incluidos en el modelo.

In [ ]:
df_sin_cliente = df[df["Name"] != "Cliente 1"]

In [ ]:
df_cat = df_sin_cliente[df_sin_cliente["category name"] == "name 1"]

In [ ]:
#Crear serie de tiempo
df_ts = df_cat.groupby("fecha")["quantity shipped"].sum().sort_index()

In [ ]:
df_ts.plot()

In [ ]:
#Crear variables
df_ts = df_cat.groupby("fecha")["quantity shipped"].sum().to_frame()

df_ts["lag1"] = df_ts["quantity shipped"].shift(1)
df_ts["lag2"] = df_ts["quantity shipped"].shift(2)
df_ts["lag3"] = df_ts["quantity shipped"].shift(3)

df_ts = df_ts.dropna()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = df_ts[["lag1", "lag2", "lag3"]]
y = df_ts["quantity shipped"]

model = RandomForestRegressor()
model.fit(X, y)

In [ ]:
y_pred = model.predict(X)

In [ ]:
#Separar datos
train = df_ts.iloc[:-6]   # todo menos últimos 6 meses
test = df_ts.iloc[-6:]    # últimos 6 meses

In [ ]:
#Definir variables
X_train = train[["lag1", "lag2", "lag3"]]
y_train = train["quantity shipped"]

X_test = test[["lag1", "lag2", "lag3"]]
y_test = test["quantity shipped"]

In [ ]:
#Entrenar modelo
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

In [ ]:
#Predecir
y_pred = model.predict(X_test)

In [ ]:
#Evaluar
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
mae

In [ ]:
#Gráfica
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(test.index, y_test, label="Real")
plt.plot(test.index, y_pred, label="Predicción")

plt.legend()
plt.title("Forecast vs Real")
plt.show()

Se observó que la presencia de un cliente con compras extraordinarias generaba alta variabilidad en la serie, afectando la precisión del modelo. Al excluir este comportamiento atípico, el error se redujo significativamente, lo que confirma que la demanda regular sigue patrones más estables.

In [ ]:
import pandas as pd

resultados = pd.DataFrame({
    "fecha": test.index,
    "real": y_test.values,
    "prediccion": y_pred
})

In [ ]:
resultados

In [ ]:
resultados["prediccion"] = resultados["prediccion"].round(0)

In [ ]:
resultados = resultados.sort_values("fecha")

In [ ]:
forecast = resultados[["fecha", "prediccion"]]

In [ ]:
ultimo = df_ts.iloc[-1]

lag1 = ultimo["quantity shipped"]
lag2 = df_ts.iloc[-2]["quantity shipped"]
lag3 = df_ts.iloc[-3]["quantity shipped"]

In [ ]:
import pandas as pd

X_new = pd.DataFrame({
    "lag1": [lag1],
    "lag2": [lag2],
    "lag3": [lag3]
})

pred_1 = model.predict(X_new)[0]

In [ ]:
predicciones = []

for i in range(6):

    X_new = pd.DataFrame({
        "lag1": [lag1],
        "lag2": [lag2],
        "lag3": [lag3]
    })

    pred = model.predict(X_new)[0]

    predicciones.append(pred)

    lag3 = lag2
    lag2 = lag1
    lag1 = pred

In [ ]:
fechas_futuras = pd.date_range(start=df_ts.index[-1], periods=7, freq="ME")[1:]

In [ ]:
forecast_futuro = pd.DataFrame({
    "fecha": fechas_futuras,
    "prediccion": predicciones
})

In [ ]:
forecast_futuro["articulo"] = "name 1"

In [ ]:
forecast_futuro

# Conclusión
A partir del análisis y modelado realizado, se logró desarrollar un enfoque de forecasting capaz de estimar la demanda de productos a nivel temporal. El modelo captura adecuadamente la tendencia general del comportamiento de ventas, permitiendo generar predicciones útiles para la planeación operativa.

Sin embargo, se identificó que la presencia de eventos atípicos, como compras extraordinarias de clientes específicos, introduce variabilidad en la serie y afecta la precisión del modelo. Al analizar escenarios sin estos eventos, se observó una mejora significativa en el desempeño, lo que evidencia que la demanda regular sigue patrones más estables.

Estos hallazgos sugieren que la demanda puede segmentarse en dos componentes: comportamiento regular y eventos extraordinarios. Por lo tanto, se recomienda complementar el modelo con información adicional relacionada con clientes o implementar estrategias diferenciadas para mejorar la precisión del forecasting.

En conjunto, este proyecto demuestra cómo el uso de machine learning puede transformar procesos manuales en herramientas analíticas que apoyan la toma de decisiones basada en datos.